# plotmux — matplotlib backend examples

This notebook shows how to use `plotmux`'s public API (`plotmux.hist`) with the matplotlib backend, and how to reach matplotlib-specific features through the `Figure` escape hatch.

In [ ]:
import numpy as np

import plotmux

rng = np.random.default_rng(42)
values = rng.normal(loc=0.0, scale=1.0, size=100_000)

## Basic histogram

`plotmux.hist` builds a `HistogramSpec` and renders it with the current default backend (`matplotlib` unless changed via `plotmux.set_backend`).

In [ ]:
fig = plotmux.hist(values, bins=101)
fig.backend_name

In [ ]:
fig.to_native()

## Custom bin count and axis range

`xmin`/`xmax` accept explicit values or quantile strings such as `"q0.1"` (10th percentile), resolved via `plotmux.core.range.find_range`.

In [ ]:
fig = plotmux.hist(values, bins=101, xmin="q0.01", xmax="q0.99")

## Probability density

Set `density=True` so the histogram integrates to 1 instead of showing raw counts.

In [ ]:
fig = plotmux.hist(values, bins=101, density=True)

## Label and legend

Passing `label` adds a legend entry, useful when overlaying multiple histograms on the same axes via `to_native()`.

In [ ]:
group_a = rng.normal(loc=0.0, scale=1.0, size=100_000)
group_b = rng.normal(loc=1.5, scale=1.0, size=100_000)

fig_a = plotmux.hist(group_a, bins=101, density=True, label="group A")
ax = fig_a.to_native().axes[0]
ax.hist(group_b, bins=101, density=True, alpha=0.6, label="group B")
ax.legend()

## Additional matplotlib keyword arguments

Extra keyword arguments are forwarded to the underlying `Axes.hist` call, so backend-specific styling is available without leaving the unified API.

In [ ]:
fig = plotmux.hist(values, bins=101, color="seagreen", alpha=0.8, histtype="stepfilled")

## Saving a figure

`Figure.save` infers the export format from the file suffix (`.png`, `.svg`, `.pdf`, ...) and delegates to the backend.

In [ ]:
fig = plotmux.hist(values, bins=101)
fig.save("../tmp/plotmux_hist.png")

## Explicit backend selection

The backend can be selected per call, set as the process-wide default, or scoped with a context manager — useful once more than one backend is registered.

In [ ]:
fig = plotmux.hist(values, bins=30, backend="matplotlib")
fig.backend_name

In [ ]:
with plotmux.backend("matplotlib"):
    fig = plotmux.hist(values, bins=30)
fig.backend_name